In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import xgboost as xgb

# 讀取數據
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv')
X_test = pd.read_csv('X_test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

# 先將 '建築完成年月' 轉換為 datetime 類型並取得年份
X_train['建築完成年月'] = pd.to_datetime(X_train['建築完成年月'], errors='coerce').dt.year
# 計算建築年齡
current_year = pd.to_datetime('today').year  # 當前年份
X_train['建築完成年月'] = current_year - X_train['建築完成年月']
# 同樣的處理 'X_test'
X_test['建築完成年月'] = pd.to_datetime(X_test['建築完成年月'], errors='coerce').dt.year
X_test['建築完成年月'] = current_year - X_test['建築完成年月']

# 填補類別型特徵的缺失值
categorical_cols = X_train.select_dtypes(include=['object']).columns

# 對所有類別型特徵進行獨熱編碼
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_encoded = onehot_encoder.fit_transform(X_train[categorical_cols])
X_test_encoded = onehot_encoder.transform(X_test[categorical_cols])

# 將編碼後的結果轉為 DataFrame 並確保列名稱為字符串
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=onehot_encoder.get_feature_names_out(categorical_cols).astype(str))
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=onehot_encoder.get_feature_names_out(categorical_cols).astype(str))

# 刪除原始類別特徵，並添加編碼後的數據
X_train = X_train.drop(categorical_cols, axis=1).reset_index(drop=True)
X_test = X_test.drop(categorical_cols, axis=1).reset_index(drop=True)

X_train = pd.concat([X_train, X_train_encoded_df], axis=1)
X_test = pd.concat([X_test, X_test_encoded_df], axis=1)

# 創建交互特徵
X_train['面積_房交互'] = X_train['土地移轉總面積平方公尺'] * X_train['建物現況格局-房']
X_test['面積_房交互'] = X_test['土地移轉總面積平方公尺'] * X_test['建物現況格局-房']
X_train['面積_地鐵站交互'] = X_train['土地移轉總面積平方公尺'] * X_train['地鐵站']
X_test['面積_地鐵站交互'] = X_test['土地移轉總面積平方公尺'] * X_test['地鐵站']
X_train['面積_廳交互'] = X_train['土地移轉總面積平方公尺'] * X_train['建物現況格局-廳']
X_test['面積_廳交互'] = X_test['土地移轉總面積平方公尺'] * X_test['建物現況格局-廳']
X_train['廳_房交互'] = X_train['建物現況格局-廳'] * X_train['建物現況格局-房']
X_test['廳_房交互'] = X_test['建物現況格局-廳'] * X_test['建物現況格局-房']
X_train['面積_橫坐標交互'] = X_train['土地移轉總面積平方公尺'] * X_train['橫坐標']
X_test['面積_橫坐標交互'] = X_test['土地移轉總面積平方公尺'] * X_test['橫坐標']

# 設置目標變數
y_train_values = y_train['單價元平方公尺'].values

# 分割數據集
X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
    X_train, y_train_values, test_size=0.2, random_state=42
)

# 標準化數據
scaler = StandardScaler()
X_train_split_scaled = scaler.fit_transform(X_train_split)
X_valid_split_scaled = scaler.transform(X_valid_split)
X_test_scaled = scaler.transform(X_test)

# 初始化變量來存儲最佳參數和最低RMSE
best_params = None
best_rmse = float('inf')

# 設置你想測試的參數
param_sets = [
    # {'n_estimators': 1500, 'max_depth': 7, 'learning_rate': 0.04, 'colsample_bytree': 0.7, 'subsample': 0.8},
    {'n_estimators': 1500, 'max_depth': 7, 'learning_rate': 0.04, 'colsample_bytree': 0.5, 'subsample': 0.8},
]

# 手動測試每組參數
for params in param_sets:
    # 添加 XGBoost 所需的目標設定
    params.update({'objective': 'reg:squarederror', 'random_state': 42})
    
    # 建立並訓練模型
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train_split_scaled, y_train_split)

    # 預測並計算驗證集 RMSE
    y_valid_pred = xgb_model.predict(X_valid_split_scaled)
    rmse = np.sqrt(mean_squared_error(y_valid_split, y_valid_pred))
    
    print(f"測試參數: {params}, 驗證集 RMSE: {rmse}")

    # 更新最佳參數
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params

# 合併訓練集與驗證集
X_full_train = np.vstack((X_train_split_scaled, X_valid_split_scaled))  # 合併 scaled 的資料
y_full_train = np.hstack((y_train_split, y_valid_split))  # 合併目標變數

# 使用最佳參數訓練最終模型
xgb_model_best = xgb.XGBRegressor(**best_params)
xgb_model_best.fit(X_full_train, y_full_train)

# 查看特徵重要性
feature_importances = xgb_model_best.feature_importances_

# 根據特徵重要性選擇最重要的特徵
threshold = 0.0008  # 設定閾值，只選擇重要性大於閾值的特徵
important_features = X_train.columns[feature_importances > threshold]

# 根據選擇的特徵來縮小數據集
X_train_important = X_train_split[important_features]  # 使用 X_train_split 進行特徵選擇
X_valid_split_important = X_valid_split[important_features]
X_test_important = X_test[important_features]

# 使用選擇後的特徵訓練模型
xgb_model_best.fit(X_train_important, y_train_split,)

# 在驗證集上預測並計算 RMSE
y_valid_pred_important = xgb_model_best.predict(X_valid_split_important)
valid_rmse = np.sqrt(mean_squared_error(y_valid_split, y_valid_pred_important))
print(f"使用選擇後特徵的驗證集 RMSE: {valid_rmse}")

# 預測測試集
y_test_pred = xgb_model_best.predict(X_test_important)

# 準備提交文件
submission = sample_submission.copy()
submission['單價元平方公尺'] = y_test_pred
submission.to_csv('submission.csv', index=False)

print("已生成 submission.csv，提交格式符合要求")